In [ ]:
import pandas as pd
df = pd.read_parquet('../data/model_data.parquet')

x_cols = [c for c in df.columns if c.startswith("x")]
cond_cols = [c for c in df.columns if c.startswith("con")]

# fill missing values in cond columns with forward fill
# backward fill is needed for the first rows if they are missing (that is lookfowarding but very few rows are missing in the beginning after forward fill)
df["cond2"] = df["cond2"].ffill()
df["cond2"] = df["cond2"].bfill()

df["cond3"] = df["cond3"].ffill()
df["cond3"] = df["cond3"].bfill()

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Optional, Dict


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def get_year_index(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s).dt.to_period("Y")


# --------------------------------------------------
# Window schedule
# --------------------------------------------------

@dataclass
class WindowSpec:
    mode: str              # "rolling" or "expanding"
    train_years: int = 4
    min_train_years: int = 4


def build_yearly_schedule(
    df: pd.DataFrame,
    date_col: str,
    window_spec: WindowSpec,
) -> List[Dict]:
    tmp = df[[date_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    years = np.array(sorted(tmp[date_col].dt.to_period("Y").unique()))

    schedule = []
    for i in range(len(years) - 1):
        pred_year = years[i + 1]

        if window_spec.mode == "rolling":
            start_idx = i - window_spec.train_years + 1
            if start_idx < 0:
                continue
            train_years = years[start_idx:i + 1]

        elif window_spec.mode == "expanding":
            if i + 1 < window_spec.min_train_years:
                continue
            train_years = years[:i + 1]

        else:
            raise ValueError("window_spec.mode must be 'rolling' or 'expanding'")

        schedule.append({
            "train_years": train_years,
            "pred_year": pred_year,
        })

    return schedule


# --------------------------------------------------
# Quantile group helpers
# --------------------------------------------------

def make_group_borders_from_train(values: np.ndarray, n_groups: int) -> np.ndarray:
    """
    Build equal-size quantile borders from training data only.
    """
    probs = np.linspace(0.0, 1.0, n_groups + 1)
    borders = np.quantile(values, probs)

    # remove duplicate borders if ties create them
    borders = np.unique(borders)

    if len(borders) < 2:
        borders = np.array([-np.inf, np.inf], dtype=float)
    else:
        borders[0] = -np.inf
        borders[-1] = np.inf

    return borders


def assign_groups_from_borders(values: np.ndarray, borders: np.ndarray) -> np.ndarray:
    """
    Assign each value to a group using the training borders.
    Outside values go to the nearest edge group automatically.
    """
    inner = borders[1:-1]
    groups = np.searchsorted(inner, values, side="right")
    return groups.astype(int)


# --------------------------------------------------
# Data-shared ridge fit/predict
# --------------------------------------------------

def fit_data_shared_ridge(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    ridge_alpha_shared: float,
    ridge_alpha_group: float,
    jitter: float = 1e-10,
):
    """
    Fit:
        (1/N) || y - X b_shared - X d_group ||^2
        + alpha_shared * ||b_shared||^2
        + alpha_group * sum_g (N_g/N) ||d_g||^2

    No intercept.
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups, dtype=int)

    n, p = X.shape
    n_groups = int(groups.max()) + 1

    # Build design matrix: [X, X*1[g=0], X*1[g=1], ..., X*1[g=G-1]]
    Z_parts = [X]
    for g in range(n_groups):
        mask = (groups == g).astype(float).reshape(-1, 1)
        Z_parts.append(X * mask)
    Z = np.hstack(Z_parts)

    # Group sizes
    group_counts = np.bincount(groups, minlength=n_groups).astype(float)

    # Solve:
    # (Z'Z / N + Lambda) theta = Z'y / N
    A = (Z.T @ Z) / n
    b = (Z.T @ y) / n

    total_dim = p * (1 + n_groups)
    penalty = np.zeros(total_dim, dtype=float)

    # shared block penalty
    penalty[:p] = ridge_alpha_shared

    # group blocks penalty
    for g in range(n_groups):
        start = p * (1 + g)
        end = start + p
        penalty[start:end] = ridge_alpha_group * (group_counts[g] / n)

    A.flat[::A.shape[0] + 1] += penalty + jitter

    theta = np.linalg.solve(A, b)

    coef_shared = theta[:p]
    coef_group = theta[p:].reshape(n_groups, p)

    return coef_shared, coef_group

def predict_data_shared_ridge(
    X: np.ndarray,
    groups: np.ndarray,
    coef_shared: np.ndarray,
    coef_group: np.ndarray,
    eps: float = 1e-8,
):
    X = np.asarray(X, dtype=float)
    groups = np.asarray(groups, dtype=int)

    # get group-specific coefs per row
    group_coefs = coef_group[groups]

    # combined coefficients per row
    combined_coefs = coef_shared + group_coefs  # shape (n_samples, n_features)

    # predictions
    pred = np.sum(X * combined_coefs, axis=1)

    # normalization: L1 norm of combined coefficients
    norm = np.sum(np.abs(combined_coefs), axis=1)

    pred /= (norm + eps)

    pred_shared = (X @ coef_shared) / (norm + eps)
    pred_group = np.sum(X * group_coefs, axis=1) / (norm + eps)

    return pred, pred_shared, pred_group

# --------------------------------------------------
# Walk-forward yearly predictions
# --------------------------------------------------

def walk_forward_yearly_predictions_data_shared_ridge(
    df: pd.DataFrame,
    x_cols: List[str],
    cond_col: str,
    n_groups: int = 3,
    target_col: str="ret_fopen",
    date_col: str = "msgStamp",
    trade_date_col: str = "trade_date",
    window_spec: WindowSpec = WindowSpec(mode="expanding", train_years=4),
    ridge_alpha_shared: float = 1.0,
    ridge_alpha_group: float = 1.0,
    x_clip: Optional[float] = 3.0,
    y_clip_quantile: Optional[float] = 0.01,
) -> pd.DataFrame:

    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])
    data = data.sort_values(date_col).reset_index(drop=True)
    data["year"] = data[date_col].dt.to_period("Y")

    if trade_date_col not in data.columns:
        data[trade_date_col] = data[date_col].dt.date

    schedule = build_yearly_schedule(data, date_col, window_spec)
    outputs = []

    for step in schedule:
        train_years = step["train_years"]
        pred_year = step["pred_year"]

        train_df = data.loc[data["year"].isin(train_years)].copy()
        test_df = data.loc[data["year"] == pred_year].copy()

        train_df = train_df.dropna(subset=x_cols + [cond_col] + [target_col]).copy()
        test_df = test_df.dropna(subset=x_cols + [cond_col]).copy()

        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_train = train_df[x_cols].to_numpy(dtype=float)
        X_test = test_df[x_cols].to_numpy(dtype=float)
        y_train = train_df[target_col].to_numpy(dtype=float)

        if y_clip_quantile is not None:
            low = np.quantile(y_train, y_clip_quantile)
            high = np.quantile(y_train, 1.0 - y_clip_quantile)
            y_train = np.clip(y_train, low, high)

        if x_clip is not None:
            X_train = np.clip(X_train, -x_clip, x_clip)
            X_test = np.clip(X_test, -x_clip, x_clip)


        cond_train = train_df[cond_col].to_numpy(dtype=float)
        cond_test = test_df[cond_col].to_numpy(dtype=float)

        borders = make_group_borders_from_train(cond_train, n_groups)
        group_train = assign_groups_from_borders(cond_train, borders)
        group_test = assign_groups_from_borders(cond_test, borders)

        coef_shared, coef_group = fit_data_shared_ridge(
            X=X_train,
            y=y_train,
            groups=group_train,
            ridge_alpha_shared=ridge_alpha_shared,
            ridge_alpha_group=ridge_alpha_group,
        )

        pred_test, pred_shared_test, pred_group_test = predict_data_shared_ridge(
            X=X_test,
            groups=group_test,
            coef_shared=coef_shared,
            coef_group=coef_group,
        )

        out = test_df[[date_col, trade_date_col]].copy()
        out["pred"] = pred_test
        out["pred_shared"] = pred_shared_test
        out["pred_group"] = pred_group_test
        out["group"] = group_test

        out["pred_year"] = str(pred_year)
        out["train_start_year"] = str(train_years[0])
        out["train_end_year"] = str(train_years[-1])

        outputs.append(out)

    if not outputs:
        return pd.DataFrame(columns=[date_col, trade_date_col, "pred"])

    return pd.concat(outputs, axis=0).sort_values(date_col).reset_index(drop=True)

In [ ]:
# --------------------------------------------------
# Backtest function 
# --------------------------------------------------
def backtest(df, feature_col, ret_col, date_col="trade_date", annualization=252):
    data = df[[date_col, feature_col, ret_col]].dropna().copy()
    data[date_col] = pd.to_datetime(data[date_col]).dt.date

    def daily_stats(group):
        alphas = group[feature_col].clip(-3, 3)
        returns = group[ret_col]
        total_pnl = np.sum(alphas * returns)
        total_gross = np.sum(np.abs(alphas))
        return pd.Series({"daily_pnl": total_pnl, "daily_gross": total_gross})

    daily_stats_df = (
        data.groupby(date_col, sort=True)
        .apply(daily_stats)
        .dropna()
    )

    daily_pnl = daily_stats_df["daily_pnl"]
    daily_gross = daily_stats_df["daily_gross"]

    cumulative_returns = daily_pnl.cumsum()
    sharpe_ratio = np.sqrt(annualization) * daily_pnl.mean() / daily_pnl.std()
    average_return = daily_pnl.sum()/daily_gross.sum() if daily_gross.sum() > 0 else 0.0

    return {
        "cumulative_returns": cumulative_returns,
        "sharpe_ratio": sharpe_ratio,
        "average_return": average_return,
    }



In [ ]:
# --------------------------------------------------
# cond1 pooling
# --------------------------------------------------

In [ ]:
ridge_alpha_shared_grid = np.array([1e-4, 1e-3, 1e-2 ,1e-1, 1, 1e1, 1e2])
ridge_alpha_group_grid = np.array([1e-4, 1e-3, 1e-2, 1e-1,  1, 1e1, 1e2])

preds = {}
search_rows = []

for alpha_shared in ridge_alpha_shared_grid:
    for alpha_group in ridge_alpha_group_grid:
        key = (float(alpha_shared), float(alpha_group))

        pred_df = walk_forward_yearly_predictions_data_shared_ridge(
            df=df,
            x_cols=x_cols,
            cond_col="cond1",
            ridge_alpha_shared=float(alpha_shared),
            ridge_alpha_group=float(alpha_group),
        )

        preds[key] = pred_df


In [ ]:
import pickle
with open('preds_shared_cond1.pkl', 'wb') as file:
    pickle.dump(preds, file)

In [ ]:
import datetime
grid_backtests = {}
grid_cumulative_pnl = {}
grid_metrics = []

for key, pred_df in preds.items():
    alpha_shared, alpha_group = key
    dd = pred_df.merge(df[["msgStamp", "ret_fopen"]], on="msgStamp", how="left")
    bt = backtest(dd[(dd["trade_date"] < datetime.date(2023, 1, 1))], feature_col="pred", ret_col="ret_fopen")

    grid_backtests[key] = bt
    grid_cumulative_pnl[key] = bt["cumulative_returns"]

    grid_metrics.append(
        {
            "ridge_alpha_shared": alpha_shared,
            "ridge_alpha_group": alpha_group,
            "sharpe_ratio": bt["sharpe_ratio"],
            "return": bt["average_return"],
        }
    )



In [ ]:
grid_metrics_df = (
    pd.DataFrame(grid_metrics)
    .sort_values(["sharpe_ratio", "return","ridge_alpha_shared", "ridge_alpha_group"], ascending=[False,False, True, True])
    .reset_index(drop=True)
)

In [ ]:
sharpe_table = grid_metrics_df.pivot(
    index="ridge_alpha_shared",
    columns="ridge_alpha_group",
    values="sharpe_ratio",
)

sharpe_table


In [ ]:
return_table = grid_metrics_df.pivot(
    index="ridge_alpha_shared",
    columns="ridge_alpha_group",
    values="return",
)

return_table


In [ ]:
# --------------------------------------------------
# cond2 pooling
# --------------------------------------------------

In [ ]:
ridge_alpha_shared_grid = np.array([1e-4, 1e-3, 1e-2 ,1e-1, 1, 1e1, 1e2])
ridge_alpha_group_grid = np.array([1e-4, 1e-3, 1e-2, 1e-1,  1, 1e1, 1e2])

preds = {}
search_rows = []

for alpha_shared in ridge_alpha_shared_grid:
    for alpha_group in ridge_alpha_group_grid:
        key = (float(alpha_shared), float(alpha_group))

        pred_df = walk_forward_yearly_predictions_data_shared_ridge(
            df=df,
            x_cols=x_cols,
            cond_col="cond2",
            ridge_alpha_shared=float(alpha_shared),
            ridge_alpha_group=float(alpha_group),
        )

        preds[key] = pred_df


In [ ]:
import pickle
with open('preds_shared_cond2.pkl', 'wb') as file:
    pickle.dump(preds, file)

In [ ]:
import datetime
grid_backtests = {}
grid_cumulative_pnl = {}
grid_metrics = []

for key, pred_df in preds.items():
    alpha_shared, alpha_group = key
    dd = pred_df.merge(df[["msgStamp", "ret_fopen"]], on="msgStamp", how="left")
    bt = backtest(dd[(dd["trade_date"] < datetime.date(2023, 1, 1))], feature_col="pred", ret_col="ret_fopen")

    grid_backtests[key] = bt
    grid_cumulative_pnl[key] = bt["cumulative_returns"]

    grid_metrics.append(
        {
            "ridge_alpha_shared": alpha_shared,
            "ridge_alpha_group": alpha_group,
            "sharpe_ratio": bt["sharpe_ratio"],
        }
    )



In [ ]:
grid_metrics_df = (
    pd.DataFrame(grid_metrics)
    .sort_values(["sharpe_ratio", "ridge_alpha_shared", "ridge_alpha_group"], ascending=[False, True, True])
    .reset_index(drop=True)
)

grid_metrics_df.head()


In [ ]:
sharpe_table = grid_metrics_df.pivot(
    index="ridge_alpha_shared",
    columns="ridge_alpha_group",
    values="sharpe_ratio",
)

sharpe_table


In [ ]:
# --------------------------------------------------
# cond3 pooling
# --------------------------------------------------

In [ ]:
ridge_alpha_shared_grid = np.array([1e-4, 1e-3, 1e-2 ,1e-1, 1, 1e1, 1e2])
ridge_alpha_group_grid = np.array([1e-4, 1e-3, 1e-2, 1e-1,  1, 1e1, 1e2])

preds = {}
search_rows = []

for alpha_shared in ridge_alpha_shared_grid:
    for alpha_group in ridge_alpha_group_grid:
        key = (float(alpha_shared), float(alpha_group))

        pred_df = walk_forward_yearly_predictions_data_shared_ridge(
            df=df,
            x_cols=x_cols,
            cond_col="cond3",
            ridge_alpha_shared=float(alpha_shared),
            ridge_alpha_group=float(alpha_group),
        )

        preds[key] = pred_df


In [ ]:
import pickle
with open('preds_shared_cond3.pkl', 'wb') as file:
    pickle.dump(preds, file)

In [ ]:
import datetime
grid_backtests = {}
grid_cumulative_pnl = {}
grid_metrics = []

for key, pred_df in preds.items():
    alpha_shared, alpha_group = key
    dd = pred_df.merge(df[["msgStamp", "ret_fopen"]], on="msgStamp", how="left")
    bt = backtest(dd[(dd["trade_date"] < datetime.date(2023, 1, 1))], feature_col="pred", ret_col="ret_fopen")

    grid_backtests[key] = bt
    grid_cumulative_pnl[key] = bt["cumulative_returns"]

    grid_metrics.append(
        {
            "ridge_alpha_shared": alpha_shared,
            "ridge_alpha_group": alpha_group,
            "sharpe_ratio": bt["sharpe_ratio"],
        }
    )



In [ ]:
grid_metrics_df = (
    pd.DataFrame(grid_metrics)
    .sort_values(["sharpe_ratio", "ridge_alpha_shared", "ridge_alpha_group"], ascending=[False, True, True])
    .reset_index(drop=True)
)

grid_metrics_df.head()


In [ ]:
sharpe_table = grid_metrics_df.pivot(
    index="ridge_alpha_shared",
    columns="ridge_alpha_group",
    values="sharpe_ratio",
)

sharpe_table


In [ ]:
# compare all the results together

In [ ]:
import pickle
preds0=[]

for i in range(1,4):
    with open(f'preds_shared_cond{i}.pkl', 'rb') as file:
        preds0.append(pickle.load(file)[(0.001, 0.1)][["msgStamp","trade_date","pred"]].rename(columns={"pred":f"pred_cond{i}"}).set_index(["msgStamp","trade_date"]))

preds = pd.concat(preds0, axis=1).sort_index()

In [ ]:
preds["pred_avg"] = preds.mean(axis=1)

In [ ]:
import datetime

signal_cols = preds.columns.tolist()
bt_input = (
    preds.reset_index()
    .merge(df[["msgStamp", "ret_fopen"]], on="msgStamp", how="left")
)

end_date = datetime.date(2023, 1, 1)
bt_input = bt_input[(bt_input["trade_date"] < end_date)].copy()
all_dates = pd.Index(sorted(bt_input["trade_date"].dropna().unique()), name="trade_date")


In [ ]:
cum_pnl_df = pd.DataFrame(index=all_dates)
sharpe_rows = []

for col in signal_cols:
    bt = backtest(bt_input, feature_col=col, ret_col="ret_fopen")
    cum_pnl_df[col] = bt["cumulative_returns"].reindex(all_dates)
    sharpe_rows.append({"signal": col, "sharpe_ratio": bt["sharpe_ratio"], "return": bt["average_return"]})

sharpe_df = pd.DataFrame(sharpe_rows).sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)


In [ ]:
sharpe_df

In [ ]:
preds.corr()

In [ ]:
ax = cum_pnl_df.plot(figsize=(12, 6), title="Cumulative PnL by signal")
ax.axhline(0, color="black", linewidth=1, alpha=0.5)
ax.set_ylabel("Cumulative PnL")